In [1]:
import pandas as pd
import numpy as np

In [2]:
np.random.seed(42)

In [11]:
patients = pd.read_csv('healthcare_dataset.csv')

In [2]:
campaigns = pd.DataFrame({
    'campaign_id': ['c1','c2','c3','c4'],
    'campaign_type': ['Awareness', 'Conversion', 'Engagement', 'Conversion'],
    'channel_type': ['Email', 'SMS','Doctor Outreach', 'Digital Ads'],
    'target_condition': ['Diabetes', 'Cancer', 'General', 'Hypertension'],
    'budget': [400000, 900000, 300000, 800000]
})

print(campaigns)

  campaign_id campaign_type     channel_type target_condition  budget
0          c1     Awareness            Email         Diabetes  400000
1          c2    Conversion              SMS           Cancer  900000
2          c3    Engagement  Doctor Outreach          General  300000
3          c4    Conversion      Digital Ads     Hypertension  800000


In [6]:
campaigns.to_csv('campaigns.csv', index=False)

In [12]:
patients['patient_id']=['p' + str(i) for i in range(len(patients))]

In [14]:
patients.to_csv('patients.csv', index=False)

In [47]:
n=100000

engagement = pd.DataFrame({
    'patient_id': np.random.choice(patients['patient_id'],size=n),
    'campaign_id': np.random.choice(campaigns['campaign_id'],size=n)
})

In [48]:
engagement.head()

,patient_id,campaign_id
0,p14707,c2
1,p46774,c2
2,p47413,c4
3,p6984,c2
4,p7797,c1


In [49]:
engagement = engagement.merge(
    patients[['patient_id', 'Medical Condition']],
    on = 'patient_id',
    how = 'left')

In [51]:
engagement['engaged'] = np.where(
((engagement['Medical Condition'] == 'Diabetes') & (engagement['campaign_id'] == 'c1')) |
((engagement['Medical Condition'] == 'Cancer') & (engagement['campaign_id'] == 'c2')) |
((engagement['Medical Condition'] == 'Hypertension') & (engagement['campaign_id'] == 'c4')),
np.random.choice([0,1], size=len(engagement), p=[0.3,0.7]),
np.random.choice([0,1], size=len(engagement), p=[0.6,0.4])
)                                 

In [52]:
engagement['engaged'].value_counts(normalize=True)

0    0.56381
1    0.43619
Name: engaged, dtype: float64

In [53]:
engagement['converted'] = np.where(
((engagement['Medical Condition'] == 'Diabetes') & (engagement['campaign_id'] == 'c1')) |
((engagement['Medical Condition'] == 'Cancer') & (engagement['campaign_id'] == 'c2')) |
((engagement['Medical Condition'] == 'Hypertension') & (engagement['campaign_id'] == 'c4')),

engagement['engaged']* np.random.choice([0,1], size=len(engagement), p=[0.5,0.5]),
    
np.where(
engagement['campaign_id'] == 'c3',
engagement['engaged']* np.random.choice([0,1], size=len(engagement), p=[0.7,0.3]),
engagement['engaged']* np.random.choice([0,1], size=len(engagement), p=[0.85,0.15])
)
)                       

In [54]:
engagement['converted'].value_counts(normalize=True)

0    0.88966
1    0.11034
Name: converted, dtype: float64

In [55]:
engagement["revenue_generated"] = engagement["converted"] * np.random.randint(
    100, 1000, size=len(engagement)
)

In [57]:
engagement.head(10)

,patient_id,campaign_id,Medical Condition,engaged,converted,revenue_generated
0,p14707,c2,Arthritis,0,0,0
1,p46774,c2,Diabetes,0,0,0
2,p47413,c4,Arthritis,0,0,0
3,p6984,c2,Cancer,1,1,420
4,p7797,c1,Cancer,1,0,0
5,p43298,c1,Diabetes,1,0,0
6,p52555,c1,Hypertension,0,0,0
7,p27173,c4,Obesity,1,0,0
8,p49935,c1,Arthritis,1,0,0
9,p41310,c1,Diabetes,1,1,966


In [59]:
engagement.isnull().sum()

patient_id           0
campaign_id          0
Medical Condition    0
engaged              0
converted            0
revenue_generated    0
dtype: int64

In [58]:
engagement.to_csv("healthcare_marketing_engagement.csv", index=False)